# Task #1: Data Loading & Normalization

In [ ]:
import requests
import hashlib
import pandas as pd

# Accessing Data

To remain compatible with Google Colab and personal development IDE's, data can be loaded by using requests. As data gets manipulated and saved over the course of this project, it is even more crucial to have one source of truth for every dataset.

In [ ]:
def load_json_from_github(path_from_root: str, branch: str="main"):
    """
    Retrieves JSON files from the Accenture 1O repository
    :param path_from_root: the path to the file from the branch's root
    :param branch: the branch the file is located on, assumes "main" branch
    :return: the file as a json object
    """
    url = f"https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1O-contract-review-challenge/{branch}/{path_from_root}"

    response = requests.get(url)
    response.raise_for_status()
    return response.json()

data = load_json_from_github(
    path_from_root="data/cuad/train_separate_questions.json",
    branch="setup-and-explore"
)
raw_data = data["data"]


# Normalizing Data

This is the first step for data preprocessing. In order to run NLP and ML techniques on the CUAD dataset, it must be normalized into a standardized structure. This structure is as follows:
```python
{
    "contracts": pd.DataFrame,
    "documents": pd.DataFrame,
    "categories": pd.DataFrame,
    "annotation_sets": pd.DataFrame,
    "spans": pd.DataFrame,
}
```
Contracts include...
- `contract_id`: an enumerated key
- `title`: the title of the contract

Documents include...
- `document_id`: an enumerated key
- `contract_id`: foreign key linked to `contracts.contract_id`
- `context`: the complete, unchanged contract text
- `context_group_id`: a hash code for the context

Categories include...
- `category_id`: the category name in snake case
- `category_name`: a human-readable formatted `category_id`
- `question`: the question asked to identify key clauses

Annotation Sets include...
- `annotation_set_id`: a key consisting of the `contract_id` and `category_id`
- `contract_id`: foreign key linked to `contracts.contract_id`
- `category_id`: foreign key linked to `categories.category_id`
- `is_impossible`: boolean indicating whether an answer exists to the asked question

Spans include...
- `span_id`: a key consisting of the `contract_id` and `category_id` and span number
- `annotation_set_id`: a foreign key linked to `annotation_sets.annotation_set_id`
- `source_qa_id`: the contract's original `title` and `category_id`
- `answer_text`: the identified clause in plain text
- `answer_start`: the starting index of the `answer_text`
- `answer_end`: the ending index of the `answer_text`

In [ ]:
def normalize_cuad(data: dict) -> dict:
    """
    Normalizes the given CUAD dataset from nested JSON objects into separate actionable DataFrames
    :param data: list of document objects from the CUAD dataset
    :return: a single dictionary that contains separate DataFrames for contracts, documents, categories, annotation sets, and spans. These DataFrames are connected via 'foreign keys' (aka their IDs)
    """

    # assign contract ids for each contract in the dataset
    for i, item in enumerate(data):
        item["contract_id"] = f"contract_{i + 1:04d}"

    # get each document and their respective data (title, context, qas)
    documents = pd.json_normalize(
        data,
        record_path="paragraphs",
        meta=["title", "contract_id"]
    )[["contract_id", "title", "context"]]

    # assign ids and hashes
    documents["document_id"] = [f"document_{i + 1:04d}" for i in range(len(documents))]
    documents["context_group_id"] = documents["context"].apply(
        lambda c: "context_" + hashlib.md5(c.encode("utf-8")).hexdigest()[:10]
    )

    # create the contracts "table" and remove duplicates
    contracts = (
        documents[["contract_id", "title"]]
        .drop_duplicates(subset="contract_id")
        .reset_index(drop=True)
    )

    # ensures all indexes are correct
    documents = documents[
        ["document_id", "contract_id", "context", "context_group_id"]
    ].reset_index(drop=True)

    # get all Q&A objects
    qas = pd.json_normalize(
        data,
        record_path=["paragraphs", "qas"],
        meta=["title", "contract_id"]
    )

    qas["category_name"] = qas["question"].str.extract(f'"([^"]+)')
    qas["category_id"] = (
        qas["category_name"]
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    qas["annotation_set_id"] = qas["contract_id"] + "__" + qas['category_id'].str.lower()

    # get categories. each question typically corresponds to a category
    categories = (
        qas[["category_id", "category_name", "question"]]
        .drop_duplicates(subset="category_id")
        .reset_index(drop=True)
    )

    # get the annotation set: one contract paired with one clause category/question
    annotation_sets = (
        qas.groupby(
            ["annotation_set_id", "contract_id", "category_id"]
        , as_index=False)["is_impossible"]
        .all()
    )

    # get each span: one individual answer inside the Q&A's answer list
    spans = qas[["answers", "id", "annotation_set_id"]].explode("answers")
    spans = spans[spans["answers"].notna()].copy() # drops impossible rows

    # expand the answer fields to be included in the top-level columns
    answer_fields = pd.json_normalize(spans["answers"])
    spans = pd.concat([spans.drop(columns="answers"), answer_fields], axis=1)

    spans = spans.rename(columns={"id": "source_qa_id", "text": "answer_text"})
    spans["answer_end"] = spans["answer_start"] + spans["answer_text"].str.len()

    # create the span id
    num_spans = spans.groupby("annotation_set_id").cumcount()
    spans["span_id"] = (
        spans["annotation_set_id"] + "__span_" + num_spans.astype(str).str.zfill(3)
    )
    spans = spans[[
        "span_id",
        "annotation_set_id",
        "source_qa_id",
        "answer_text",
        "answer_start",
        "answer_end",
    ]].reset_index(drop=True)

    return {
        "contracts": contracts,
        "documents": documents,
        "categories": categories,
        "annotation_sets": annotation_sets,
        "spans": spans,
    }



# Demo/Visualization

Recommended to use PyCharm or an equivalent IDE plugin to see DataFrame statistics.

In [ ]:
for i, item in enumerate(raw_data):
    item["contract_id"] = f"contract_{i + 1:04d}"

In [ ]:
df_documents = pd.json_normalize(
    raw_data,
    record_path="paragraphs",
    meta=["title", "contract_id"]
)[["contract_id", "title", "context"]]
df_documents

In [ ]:
df_documents["document_id"] = [f"document_{i + 1:04d}" for i in range(len(df_documents))]
df_documents["context_group_id"] = df_documents["context"].apply(
    lambda c: "context_" + hashlib.md5(c.encode("utf-8")).hexdigest()[:10]
)
df_documents

In [ ]:
df_contracts = (
    df_documents[["contract_id", "title"]]
    .drop_duplicates(subset="contract_id")
    .reset_index(drop=True)
)
df_contracts

In [ ]:
df_documents = df_documents[
    ["document_id", "contract_id", "context", "context_group_id"]
].reset_index(drop=True)
df_documents

In [ ]:
df_qas = pd.json_normalize(
    raw_data,
    record_path=["paragraphs", "qas"],
    meta=["title", "contract_id"]
)
df_qas

In [ ]:
df_qas["category_name"] = df_qas["question"].str.extract(f'"([^"]+)')
df_qas["category_id"] = (
    df_qas["category_name"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
df_qas["annotation_set_id"] = df_qas["contract_id"] + "__" + df_qas['category_id'].str.lower()

df_qas

In [ ]:
df_categories = (
    df_qas[["category_id", "category_name", "question"]]
    .drop_duplicates(subset="category_id")
    .reset_index(drop=True)
)

df_categories

In [ ]:
df_annotation_sets = (
    df_qas.groupby(
        ["annotation_set_id", "contract_id", "category_id"]
    , as_index=False)["is_impossible"]
    .all()
)

df_annotation_sets

In [ ]:
df_spans = df_qas[["answers", "id", "annotation_set_id"]].explode("answers")
df_spans = df_spans[df_spans["answers"].notna()].copy() # drops impossible rows
df_spans

In [ ]:
df_answer_fields = pd.json_normalize(df_spans["answers"])
df_answer_fields

In [ ]:
df_spans = pd.concat([df_spans.drop(columns="answers"), df_answer_fields], axis=1)
df_spans = df_spans.rename(columns={"id": "source_qa_id", "text": "answer_text"})
df_spans["answer_end"] = df_spans["answer_start"] + df_spans["answer_text"].str.len()
df_spans

In [ ]:
ag_num_spans = df_spans.groupby("annotation_set_id").cumcount()
df_spans["span_id"] = (
        df_spans["annotation_set_id"] + "__span_" + ag_num_spans.astype(str).str.zfill(3)
)
df_spans = df_spans[[
    "span_id",
    "annotation_set_id",
    "source_qa_id",
    "answer_text",
    "answer_start",
    "answer_end",
]].reset_index(drop=True)
df_spans

# Verifying Counts

In [ ]:
normalized_data = normalize_cuad(raw_data)

In [ ]:
totalContracts = 408
totalDocuments = 408
totalCategories = 41
totalAnnotationSets = 16728
totalSpans = 11180

contracts = normalized_data["contracts"]
documents = normalized_data["documents"]
categories = normalized_data["categories"]
annotation_sets = normalized_data["annotation_sets"]
spans = normalized_data["spans"]

Confirm these exact totals:
- [x] 408 Contracts
- [x] 408 Documents
- [x] 41 Categories
- [x] 16,728 Annotation Sets
- [x] 11,180 Spans

In [ ]:
print(f"{totalContracts} Contracts \t\t\t\t{"PASSED" if contracts.shape[0] == totalContracts else "FAILED"}")
print(f"{totalDocuments} Documents \t\t\t\t{"PASSED" if documents.shape[0] == totalDocuments else "FAILED"}")
print(f"{totalCategories} Categories \t\t\t\t{"PASSED" if categories.shape[0] == totalCategories else "FAILED"}")
print(f"{totalAnnotationSets} AnnotationSets \t\t{"PASSED" if annotation_sets.shape[0] == totalAnnotationSets else "FAILED"}")
print(f"{totalSpans} Spans \t\t\t\t{"PASSED" if spans.shape[0] == totalSpans else "FAILED"}")


# Export the data frames

parquets for use in splitting the data into training and test 80/20

In [ ]:
import pyarrow

contracts.to_parquet("contracts.parquet", engine="pyarrow", index=False)
documents.to_parquet("documents.parquet", engine="pyarrow", index=False)
categories.to_parquet("categories.parquet", engine="pyarrow", index=False)
annotation_sets.to_parquet("annotation_sets.parquet", engine="pyarrow", index=False)
spans.to_parquet("spans.parquet", engine="pyarrow", index=False)

Put parquets in their own directory to easily find

In [ ]:
from pathlib import Path

output_dir = Path("split_data")
output_dir.mkdir(exist_ok=True)

contracts.to_parquet(output_dir / "contracts.parquet", index=False)
documents.to_parquet(output_dir / "documents.parquet", index=False)
categories.to_parquet(output_dir / "categories.parquet", index=False)
annotation_sets.to_parquet(output_dir / "annotation_sets.parquet", index=False)
spans.to_parquet(output_dir / "spans.parquet", index=False)

#Verify exports

In [ ]:
print(contracts.shape)
print(documents.shape)
print(categories.shape)
print(annotation_sets.shape)
print(spans.shape)

print("Saved training-approved data.")